Task One

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly
data = pd.read_csv('Nat_Gas.csv')
data.rename(columns={'Dates': 'ds', 'Prices': 'y'},inplace=True)
data['ds'] = pd.to_datetime(data['ds'])
dd = data.set_index('ds').resample('D').interpolate(method='time')
dd.tail(50)

data['ds'] = pd.to_datetime(data['ds'])
data.set_index('ds', inplace=True)
date_range = pd.date_range(start=data.index.min(), end=data.index.max(), freq='D')
dd = data.reindex(date_range)
dd['y'] = dd['y'].interpolate(method='linear')
dd.reset_index(inplace=True)
dd.rename(columns={'index': 'ds'}, inplace=True)
plt.figure(figsize=(10, 6)) 
plt.plot(dd['ds'], dd['y'], label='Price')
plt.xlabel('Date')
plt.ylabel('Price')
plt.title('Daily Interpolated Natural Gas Prices')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
dd.tail(62)

In [ ]:
len(dd)

In [ ]:
train = dd.iloc[:len(dd)-274]
test = dd.iloc[len(dd)-274:]
print(test)
print(train)

In [ ]:
m = Prophet()
m.fit(train)
future = m.make_future_dataframe(periods = 731, freq = 'D')
forecast = m.predict(future)
forecast.loc[:,['ds','yhat']]
forecast.loc[:,['ds','yhat']]



In [ ]:
plot_components_plotly(m,forecast)

In [ ]:
from statsmodels.tools.eval_measures import rmse
predictions = forecast.iloc[1157:1431]['yhat']
print("Root mean squared error:", rmse(predictions,test['y']))
print("Mean value:",test['y'].mean())
print("Final Forecast of prices:")
fin = forecast.loc[:,['ds','yhat']].tail(457)
fin.rename(columns={'yhat': 'y'}, inplace=True)
final = pd.concat([dd,fin])
final.plot(x='ds',y='y')

In [ ]:
def findprice(input):
    # Convert the input date string to datetime
    sd = pd.to_datetime(input)
    
    # Filter the DataFrame for the specific date
    result = final[final['ds'] == sd]
    
    if not result.empty:
        return round(float(result['y'].iloc[0]),2)
    else:
        return "Date does not exist"
findprice("2025-12-31")

Task Two

In [ ]:
#Change variables here
ij_dates = ["2022-01-01","2022-02-01","2022-04-01","2022-04-01"]
wd_dates = ["2023-01-27","2022-02-15","2022-03-20",'2022-06-01']
ij_wd_rate = 1000000
max_cap = 5000000
storage_costs = 10000
ij_cost = 0.0005
wd_cost = 0.0005

In [ ]:
def fin_value():
    ij_prices = []
    wd_prices = []

    for i in ij_dates:
        ij_prices.append(findprice(i))
    print(ij_prices)

    for i in wd_dates:
        wd_prices.append(findprice(i))
    print(wd_prices)

    # Combine transactions and process them chronologically
    events = []

    for date, price in zip(ij_dates, ij_prices):
        events.append((pd.to_datetime(date), "injection", price))

    for date, price in zip(wd_dates, wd_prices):
        events.append((pd.to_datetime(date), "withdrawal", price))

    # If both happen on the same date, process injection first
    events.sort(
        key=lambda event: (
            event[0],
            0 if event[1] == "injection" else 1
        )
    )

    current_vol = 0
    value = 0

    for date, transaction, price in events:

        if transaction == "injection":
            if current_vol + ij_wd_rate > max_cap:
                print("Injection is not possible on", date.date(),
                      "because there is insufficient capacity")
                continue

            current_vol += ij_wd_rate

            # Purchase cost and injection cost
            value -= ij_wd_rate * price
            value -= ij_wd_rate * ij_cost

        if transaction == "withdrawal":
            if current_vol < ij_wd_rate:
                print("Withdrawal is not possible on", date.date(),
                      "because there is insufficient gas")
                continue

            current_vol -= ij_wd_rate

            # Sale revenue less withdrawal cost
            value += ij_wd_rate * price
            value -= ij_wd_rate * wd_cost

        print(date.date(), current_vol)

    # Charge storage from first injection to last withdrawal
    first_date = min(pd.to_datetime(ij_dates))
    last_date = max(pd.to_datetime(wd_dates))

    total_months = (
        (last_date.year - first_date.year) * 12
        + last_date.month
        - first_date.month
    )

    if last_date.day > first_date.day:
        total_months += 1

    print("Storage months:", total_months)

    value -= total_months * storage_costs

    print("Final value:")
    return round(float(value), 2)


fin_value()